# 15 — Hurdle (Two-Stage) Model

**Motivation:** `blocked_days_Q1_2026` is zero for **49.3%** of listings (no booking activity
at all in Jan-Mar 2026) — a heavily zero-inflated count. A single regressor has to learn
"is this listing active at all?" and "how many nights, given it's active?" simultaneously,
which is a harder joint problem than solving them separately.

**Hurdle model:**
1. **Classifier** `P(y>0 | x)` — will this listing get any booking activity at all?
2. **Regressor** `E[y | y>0, x]` — trained *only* on the positive-y rows — how many nights,
   given it's active?
3. **Prediction (expected value):** `y_hat = clip(P(y>0) * E[y|y>0], 0, 90)`

This mirrors `hurdle.py` from a parallel Inside Airbnb replication of this project (Ali's
branch), where the hurdle approach gave a modest but real gain: single-stage blend
MSE=256.70 -> hurdle+blend MSE=255.40 (~0.5%). That target's zero-rate was 47.7% — close
enough to this project's 49.3% that a similar-sized gain is a reasonable expectation here.

Built for XGBoost and LightGBM (both stages), with fold-level early stopping matching
`08_XGBoost_v2.ipynb` / `10_SOTA_and_diagnostics.ipynb`. The two hurdle expected-value
predictions are then blended (SLSQP) together with the existing best single-stage blend
(`blend_v2`) to see whether the hurdle framing adds anything on top of what's already there.

Outputs:
- `outputs/oof_hurdle_xgb.npy`, `outputs/oof_hurdle_lgb.npy`
- `outputs/test_local_pred_hurdle_xgb.npy`, `outputs/test_local_pred_hurdle_lgb.npy`
- `outputs/test_local_pred_hurdle_final.npy`
- `outputs/hurdle_results.txt`

In [1]:
# Notebook compatibility helper
import os
os.environ['PYTHONWARNINGS'] = 'ignore'  # also silences warnings from n_jobs=-1 joblib subprocesses
from pathlib import Path
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

## 1. Setup & Data

In [2]:
import json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.optimize import minimize
from xgboost import XGBRegressor, XGBClassifier
from lightgbm import LGBMRegressor, LGBMClassifier
import lightgbm as lgb
warnings.filterwarnings('ignore')

OUT = Path('outputs')
RS, K = 42, 5
TARGET, ID = 'blocked_days_Q1_2026', 'id'

train = pd.read_parquet(OUT / 'train_local.parquet')
test_local_df = pd.read_parquet(OUT / 'test_local.parquet')
y = train[TARGET].astype(float).values
X = train.drop(columns=[TARGET, ID]).reset_index(drop=True)
X_local_test = test_local_df.drop(columns=[TARGET, ID]).reindex(columns=X.columns).reset_index(drop=True)
y_local_test = test_local_df[TARGET].astype(float).values

print(f'Train: {X.shape} | Local test: {X_local_test.shape}')
print(f'Zero rate (train): {(y == 0).mean():.3f}')

cat_all = X.select_dtypes(exclude='number').columns.tolist()
card = {c: X[c].nunique(dropna=False) for c in cat_all}
cat_high = [c for c, n in card.items() if n > 15]
cat_low = [c for c, n in card.items() if n <= 15]
num_cols = X.select_dtypes(include='number').columns.tolist()
print(f'num={len(num_cols)}  cat_low={cat_low}  cat_high={cat_high}')

Train: (29008, 196) | Local test: (7253, 196)
Zero rate (train): 0.493
num=195  cat_low=[]  cat_high=['neighbourhood_cleansed']


## 2. Preprocessing (same pipeline as other model notebooks)

In [3]:
class KFoldTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, n_splits=5, smoothing=20.0, random_state=42):
        self.cols = cols; self.n_splits = n_splits
        self.smoothing = smoothing; self.random_state = random_state

    def _smap(self, x, yy):
        st = pd.DataFrame({'c': x, 'y': yy}).groupby('c')['y'].agg(['mean', 'count'])
        return ((st['count'] * st['mean'] + self.smoothing * self.gm_)
                / (st['count'] + self.smoothing)).to_dict()

    def fit(self, X, y):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return self

    def transform(self, X):
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = (X[c].astype(str).fillna('__nan__').map(self.maps_[c])
                       .fillna(self.gm_).astype('float32'))
        return Xo

    def fit_transform(self, X, y=None, **kw):
        y = np.asarray(y, float); self.gm_ = float(y.mean())
        Xo = X.copy()
        for c in self.cols:
            Xo[c] = np.full(len(X), self.gm_, dtype='float32')
        kf = KFold(self.n_splits, shuffle=True, random_state=self.random_state)
        for tr, va in kf.split(X):
            for c in self.cols:
                m = self._smap(X[c].astype(str).fillna('__nan__').iloc[tr], y[tr])
                Xo.iloc[va, Xo.columns.get_loc(c)] = (
                    X[c].astype(str).fillna('__nan__').iloc[va].map(m)
                       .fillna(self.gm_).astype('float32').values)
        self.maps_ = {c: self._smap(X[c].astype(str).fillna('__nan__'), y) for c in self.cols}
        return Xo


def make_pp():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median'))]), num_cols),
        ('low', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_low),
        ('high', Pipeline([('te', KFoldTargetEncoder(cat_high, 5, 20, RS))]), cat_high),
    ])

## 3. Hurdle CV Evaluator

Both stages use fold-level early stopping (10% inner-validation slice carved from each fold's
training portion, matching `08_XGBoost_v2.ipynb` / `10_SOTA_and_diagnostics.ipynb`). Stage 2
(the regressor) is fit **only on rows where `y_tr > 0`** within that fold's training portion.

In [4]:
def evaluate_hurdle(model_name, clf_ctor, reg_ctor, clf_params, reg_params,
                     return_oof=False, return_models=False):
    kf = KFold(K, shuffle=True, random_state=RS)
    fold_mses = []
    oof = np.zeros(len(y)) if return_oof else None
    models_out = []
    for fold, (tr, va) in enumerate(kf.split(X)):
        pp = make_pp()
        Xtr_full = pp.fit_transform(X.iloc[tr], y[tr])
        Xva = pp.transform(X.iloc[va])
        ytr = y[tr]
        pos = ytr > 0

        # inner validation slices for early stopping (classifier: stratified by pos/neg;
        # regressor: drawn only from the positive rows)
        Xtr_c, Xin_c, ytr_c, yin_c = train_test_split(
            Xtr_full, pos.astype(int), test_size=0.10, random_state=RS + fold, stratify=pos)
        Xtr_pos, Xin_pos, ytr_pos, yin_pos = train_test_split(
            Xtr_full[pos], ytr[pos], test_size=0.10, random_state=RS + fold)

        clf = clf_ctor(clf_params, RS + fold)
        reg = reg_ctor(reg_params, RS + fold)

        if model_name == 'LightGBM':
            clf.fit(Xtr_c, ytr_c, eval_set=[(Xin_c, yin_c)], callbacks=[lgb.early_stopping(50, verbose=False)])
            reg.fit(Xtr_pos, ytr_pos, eval_set=[(Xin_pos, yin_pos)], callbacks=[lgb.early_stopping(50, verbose=False)])
        else:  # XGBoost
            clf.fit(Xtr_c, ytr_c, eval_set=[(Xin_c, yin_c)], verbose=False)
            reg.fit(Xtr_pos, ytr_pos, eval_set=[(Xin_pos, yin_pos)], verbose=False)

        p = clf.predict_proba(Xva)[:, 1]
        e = np.clip(reg.predict(Xva), 0, 90)
        pred = np.clip(p * e, 0, 90)

        fold_mses.append(mean_squared_error(y[va], pred))
        if return_oof: oof[va] = pred
        if return_models: models_out.append((clf, reg, pp))

    return {'mse_mean': float(np.mean(fold_mses)), 'mse_std': float(np.std(fold_mses)),
            'oof': oof, 'models': models_out}

## 4. Run Hurdle — XGBoost and LightGBM

In [5]:
XGB_CLF = dict(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8,
               colsample_bytree=0.8, reg_lambda=3.0, tree_method='hist',
               n_jobs=-1, eval_metric='logloss', early_stopping_rounds=50)
XGB_REG = dict(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8,
               colsample_bytree=0.8, reg_lambda=3.0, tree_method='hist',
               n_jobs=-1, eval_metric='rmse', early_stopping_rounds=50)
LGB_CLF = dict(n_estimators=1000, learning_rate=0.05, num_leaves=48, subsample=0.8,
               colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1, verbose=-1)
LGB_REG = dict(n_estimators=1000, learning_rate=0.05, num_leaves=48, subsample=0.8,
               colsample_bytree=0.8, reg_lambda=3.0, n_jobs=-1, verbose=-1)

hurdle_configs = {
    'XGBoost': dict(
        clf_ctor=lambda p, seed: XGBClassifier(**p, random_state=seed),
        reg_ctor=lambda p, seed: XGBRegressor(**p, random_state=seed),
        clf_params=XGB_CLF, reg_params=XGB_REG,
    ),
    'LightGBM': dict(
        clf_ctor=lambda p, seed: LGBMClassifier(**p, random_state=seed),
        reg_ctor=lambda p, seed: LGBMRegressor(**p, random_state=seed),
        clf_params=LGB_CLF, reg_params=LGB_REG,
    ),
}

hurdle_results = {}
for model_name, cfg in hurdle_configs.items():
    print(f'\n>>> Hurdle [{model_name}]...')
    t0 = time.time()
    res = evaluate_hurdle(model_name, cfg['clf_ctor'], cfg['reg_ctor'],
                          cfg['clf_params'], cfg['reg_params'],
                          return_oof=True, return_models=True)
    print(f'  OOF MSE = {res["mse_mean"]:.3f} +- {res["mse_std"]:.3f} | {(time.time()-t0)/60:.1f} min')
    hurdle_results[model_name] = res
    np.save(OUT / f'oof_hurdle_{model_name.lower()[:3]}.npy', res['oof'])


>>> Hurdle [XGBoost]...
  OOF MSE = 304.985 +- 2.884 | 0.4 min

>>> Hurdle [LightGBM]...
  OOF MSE = 305.503 +- 1.946 | 0.5 min


## 5. Local Test Predictions

In [6]:
hurdle_test_preds = {}
for model_name, res in hurdle_results.items():
    preds = np.zeros(len(X_local_test))
    for clf, reg, pp in res['models']:
        Xtl = pp.transform(X_local_test)
        p = clf.predict_proba(Xtl)[:, 1]
        e = np.clip(reg.predict(Xtl), 0, 90)
        preds += np.clip(p * e, 0, 90)
    preds /= len(res['models'])
    hurdle_test_preds[model_name] = preds
    mse_t = mean_squared_error(y_local_test, preds)
    print(f'{model_name:<10} hurdle Local Test MSE = {mse_t:.3f}')
    np.save(OUT / f'test_local_pred_hurdle_{model_name.lower()[:3]}.npy', preds)

XGBoost    hurdle Local Test MSE = 303.416
LightGBM   hurdle Local Test MSE = 302.338


## 6. Blend: Hurdle EVs + Existing Single-Stage Blend

Combine the two hurdle expected-value predictions with the best single-stage blend
(`blend_v2` from `11_Blend_v2.ipynb`) via the same SLSQP convex-weight optimisation used
throughout this project, and see whether the hurdle framing adds anything on top.

In [7]:
# Load the existing single-stage blend (dev/OOF side, reconstructed from its saved weights)
with open(OUT / 'blend_v2_weights.json') as f:
    blend_v2_weights = json.load(f)

blend_v2_oof_files = {
    'LinearRegression': 'oof_LinearRegression.npy', 'RandomForest': 'oof_RandomForest.npy',
    'GradientBoosting': 'oof_GradientBoosting.npy', 'XGBoost': 'oof_XGBoost.npy',
    'XGBoost v2': 'oof_XGBoostV2.npy', 'XGBoost v3': 'oof_XGBoostV3.npy',
    'MLP': 'oof_MLP.npy', 'LightGBM': 'oof_LightGBM.npy', 'CatBoost': 'oof_CatBoost.npy',
}
names_v2 = [n for n in blend_v2_weights if (OUT / blend_v2_oof_files[n]).exists()]
M_v2 = np.column_stack([np.clip(np.load(OUT / blend_v2_oof_files[n]), 0, 90) for n in names_v2])
w_v2 = np.array([blend_v2_weights[n] for n in names_v2]); w_v2 = w_v2 / w_v2.sum()
single_blend_oof = np.clip(M_v2 @ w_v2, 0, 90)
single_blend_test = np.load(OUT / 'test_local_pred_blend_v2.npy')

print(f'Single-stage blend (v2) OOF  MSE = {mean_squared_error(y, single_blend_oof):.3f}')
print(f'Single-stage blend (v2) Test MSE = {mean_squared_error(y_local_test, single_blend_test):.3f}')

comp_oof = {'hurdle_xgb': hurdle_results['XGBoost']['oof'],
            'hurdle_lgb': hurdle_results['LightGBM']['oof'],
            'single_blend': single_blend_oof}
comp_test = {'hurdle_xgb': hurdle_test_preds['XGBoost'],
             'hurdle_lgb': hurdle_test_preds['LightGBM'],
             'single_blend': single_blend_test}

names = list(comp_oof)
M = np.column_stack([comp_oof[n] for n in names])
def bm(w): return mean_squared_error(y, np.clip(M @ w, 0, 90))
r = minimize(bm, np.full(len(names), 1/len(names)), method='SLSQP', bounds=[(0, 1)] * len(names),
             constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}], options={'ftol': 1e-9, 'maxiter': 500})
w = r.x.copy(); w[w < 1e-4] = 0; w = w / w.sum()
final_oof = np.clip(M @ w, 0, 90)

M_test = np.column_stack([comp_test[n] for n in names])
final_test = np.clip(M_test @ w, 0, 90)

print('\nFinal blend weights:', {n: round(float(wi), 4) for n, wi in zip(names, w)})
print(f'HURDLE+single blend OOF  MSE = {mean_squared_error(y, final_oof):.3f}  R2 = {r2_score(y, final_oof):.3f}')
print(f'HURDLE+single blend Test MSE = {mean_squared_error(y_local_test, final_test):.3f}  R2 = {r2_score(y_local_test, final_test):.3f}')

np.save(OUT / 'test_local_pred_hurdle_final.npy', final_test)
with open(OUT / 'hurdle_blend_weights.json', 'w') as f:
    json.dump({n: float(wi) for n, wi in zip(names, w)}, f, indent=2)

Single-stage blend (v2) OOF  MSE = 297.966
Single-stage blend (v2) Test MSE = 294.607

Final blend weights: {'hurdle_xgb': 0.101, 'hurdle_lgb': 0.1161, 'single_blend': 0.7829}
HURDLE+single blend OOF  MSE = 297.505  R2 = 0.528
HURDLE+single blend Test MSE = 295.140  R2 = 0.524


## 7. Results Summary

In [8]:
def m3(y_true, y_pred):
    return (mean_squared_error(y_true, y_pred), mean_absolute_error(y_true, y_pred), r2_score(y_true, y_pred))

rows = []
for label, oof_arr, test_arr in [
    ('hurdle_xgb_EV', hurdle_results['XGBoost']['oof'], hurdle_test_preds['XGBoost']),
    ('hurdle_lgb_EV', hurdle_results['LightGBM']['oof'], hurdle_test_preds['LightGBM']),
    ('single_blend (v2)', single_blend_oof, single_blend_test),
    ('HURDLE+single blend', final_oof, final_test),
]:
    mse_d, mae_d, r2_d = m3(y, oof_arr)
    mse_t, mae_t, r2_t = m3(y_local_test, test_arr)
    rows.append((label, mse_d, mae_d, r2_d, mse_t, mae_t, r2_t))

lines = [f'HURDLE MODEL RESULTS (target={TARGET}, 5-fold CV, n={len(y)}, zero-rate={(y==0).mean():.3f})',
         '=' * 100,
         f'{"Variant":<22}{"DevMSE":>9}{"DevMAE":>8}{"DevR2":>8}{"TestMSE":>10}{"TestMAE":>9}{"TestR2":>8}']
for label, mse_d, mae_d, r2_d, mse_t, mae_t, r2_t in rows:
    lines.append(f'{label:<22}{mse_d:>9.2f}{mae_d:>8.2f}{r2_d:>8.3f}{mse_t:>10.2f}{mae_t:>9.2f}{r2_t:>8.3f}')

rep = '\n'.join(lines)
print(rep)
(OUT / 'hurdle_results.txt').write_text(rep, encoding='utf-8')
print('\nsaved hurdle_results.txt')

HURDLE MODEL RESULTS (target=blocked_days_Q1_2026, 5-fold CV, n=29008, zero-rate=0.493)
Variant                  DevMSE  DevMAE   DevR2   TestMSE  TestMAE  TestR2
hurdle_xgb_EV            304.98   10.38   0.516    303.42    10.22   0.511
hurdle_lgb_EV            305.50   10.37   0.515    302.34    10.19   0.513
single_blend (v2)        297.97   10.21   0.527    294.61    10.01   0.525
HURDLE+single blend      297.51   10.22   0.528    295.14    10.04   0.524

saved hurdle_results.txt


## Summary
- Two-stage hurdle model (classifier `P(y>0)` + regressor `E[y|y>0]`, fit only on positive rows)
  built for XGBoost and LightGBM, both with fold-level early stopping.
- Hurdle expected-value predictions blended (SLSQP) with the existing best single-stage blend
  (`blend_v2`) — final weights and metrics saved to `hurdle_blend_weights.json` / `hurdle_results.txt`.
- Compare `HURDLE+single blend` MSE/MAE/R2 against `single_blend (v2)` in the table above: if the
  hurdle framing helps (as it did on Ali's branch, ~0.5% MSE reduction), the HURDLE+single row
  should show a lower MSE than the single_blend row alone.